# Predict and evaluate

Thin runner over `iiss.eval.batch`: every finished run on `valid`, `test` and `phone`. Each job exports the predictions of `weights/last.pt` (`runs/<run_id>/predictions/`) and scores them with pycocotools (`results/eval/<run_id>/<eval_set>/`).

- `phone` is also scored per capture condition from `metadata.csv` (`normal` and `low_light`). The classes with GT differ between conditions, so compare condition APs with care; the per-class CSV of each condition is written too.
- Needs `dataset/derived/phone/annotations.coco.json` (`python -m iiss.data.phone`).
- Can be stopped and started again: finished predictions are reused, scoring always runs again. `OVERWRITE = True` predicts again. A failing job does not stop the others; check the `status` column.
- APs/APm/APl use the `area` stored in the GT. Roboflow stores the bounding-box area.
- Close other GPU and memory-heavy programs. Phone photos are 12 MP, so 100 masks per image need more than 1 GB of RAM.
- `SMOKE = True`: S=320 checkpoints, 8 images per set, output in `runs/smoke/`.
- Keep the outputs after the real run: they are process evidence.

## Parameters

In [1]:
MODELS = ['yolo', 'maskrcnn', 'm2f']
SIZES = [320, 640, 960]
EVAL_SETS = ['valid', 'test', 'phone']
SEED = 42
OVERWRITE = False     # True: export predictions again even if they are complete
SMOKE = False         # True: forces SIZES = [320]

In [2]:
import pandas as pd
import torch

from iiss.eval.batch import condition_table, list_jobs, run_jobs, status_table
from iiss.utils.log import setup_logging

setup_logging()
pd.set_option('display.max_columns', 30, 'display.max_rows', 100, 'display.width', 200)
jobs = list_jobs(MODELS, [320] if SMOKE else SIZES, EVAL_SETS, SEED)
print(len(jobs), 'jobs;', torch.__version__,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

27 jobs; 2.14.1+cu126 NVIDIA GeForce RTX 4050 Laptop GPU


## Status before

In [3]:
status_table(jobs)

,run_id,eval_set,images,checkpoint,predictions,evaluated
0,yolo_s320_seed42,valid,166,True,False,False
1,yolo_s320_seed42,test,359,True,False,False
2,yolo_s320_seed42,phone,40,True,False,False
3,yolo_s640_seed42,valid,166,True,False,False
4,yolo_s640_seed42,test,359,True,False,False
5,yolo_s640_seed42,phone,40,True,False,False
6,yolo_s960_seed42,valid,166,True,False,False
7,yolo_s960_seed42,test,359,True,False,False
8,yolo_s960_seed42,phone,40,True,False,False
9,maskrcnn_s320_seed42,valid,166,True,False,False


## Predict and score

Interrupting the kernel is safe: run the cell again to continue.

In [4]:
table = run_jobs(jobs, smoke=SMOKE, overwrite=OVERWRITE)

2026-11-24 20:55:22 INFO    iiss.eval.batch: yolo_s320_seed42 on valid: predicting
predict valid: 100%|██████████| 166/166 [01:10<00:00,  2.35img/s]
2026-11-24 20:56:33 INFO    iiss.eval.results_json: Wrote 14177 results to C:\Users\Syauqi Nabil\Lectures\University\Semester 5\Computer Vision\Task\Instance Segmentation\indoor-instance-seg-shift\runs\yolo_s320_seed42\predictions\valid.json
2026-11-24 20:56:40 INFO    iiss.evaluate: segm AP 18.9 / bbox AP 21.8 -> C:\Users\Syauqi Nabil\Lectures\University\Semester 5\Computer Vision\Task\Instance Segmentation\indoor-instance-seg-shift\results\eval\yolo_s320_seed42\valid
2026-11-24 20:56:40 INFO    iiss.eval.batch: yolo_s320_seed42 on test: predicting
predict test: 100%|██████████| 359/359 [04:57<00:00,  1.21img/s]
2026-11-24 21:01:39 INFO    iiss.eval.results_json: Wrote 33379 results to C:\Users\Syauqi Nabil\Lectures\University\Semester 5\Computer Vision\Task\Instance Segmentation\indoor-instance-seg-shift\runs\yolo_s320_seed42\predictions

## Result

Mask (segm) AP unless the column says `bbox`. Any `failed` status is listed first.

In [5]:
failed = table[table['status'] != 'ok']
print(f'{len(failed)} of {len(table)} jobs failed')
display(failed[['run_id', 'eval_set', 'status']])
display(table[table['status'] == 'ok'].drop(columns=['status', 'out_dir'], errors='ignore'))

0 of 27 jobs failed


,run_id,eval_set,status


,run_id,eval_set,reused,AP,AP50,AP75,APs,APm,APl,AR100,bbox_AP,ms_per_image,peak_vram_mb,predict_s,eval_s
0,yolo_s320_seed42,valid,False,0.189253,0.282558,0.207047,0.033100,0.129105,0.246696,0.287865,0.217916,254.67,111.8,71.4,6.8
1,yolo_s320_seed42,test,False,0.071457,0.124066,0.071773,0.043749,0.071855,0.098683,0.177733,0.088047,417.82,113.7,298.7,8.7
2,yolo_s320_seed42,phone,False,0.054885,0.090052,0.061901,NaN,NaN,0.054890,0.128615,0.070897,3189.60,113.7,471.7,2.4
3,yolo_s640_seed42,valid,False,0.256125,0.356270,0.262988,0.093942,0.207913,0.280635,0.369871,0.283867,341.36,261.4,85.9,6.3
4,yolo_s640_seed42,test,False,0.107590,0.166930,0.113702,0.055391,0.125546,0.138614,0.219087,0.123873,543.65,278.5,296.2,5.4
5,yolo_s640_seed42,phone,False,0.072052,0.093940,0.079002,NaN,NaN,0.072981,0.146978,0.071016,5176.29,252.7,837.3,2.8
6,yolo_s960_seed42,valid,False,0.241058,0.327418,0.254209,0.119798,0.192354,0.281932,0.355816,0.263074,765.45,311.4,158.6,6.6
7,yolo_s960_seed42,test,False,0.104325,0.160992,0.109570,0.066257,0.114269,0.123648,0.228847,0.120751,986.25,291.7,503.1,14.9
8,yolo_s960_seed42,phone,False,0.052529,0.076467,0.058336,NaN,NaN,0.052542,0.120394,0.070038,5128.77,271.6,724.9,1.1
9,maskrcnn_s320_seed42,valid,False,0.133785,0.211447,0.139430,0.011805,0.093778,0.180283,0.259338,0.147697,181.32,339.8,40.2,2.9


## Phone set per condition

In [6]:
condition_table(jobs, smoke=SMOKE)

,run_id,eval_set,condition,n_images,n_gt,n_categories,AP,AP50
0,yolo_s320_seed42,phone,normal,28,123,17,0.067059,0.111103
1,yolo_s320_seed42,phone,low_light,12,43,12,0.043812,0.089934
2,yolo_s640_seed42,phone,normal,28,123,17,0.101028,0.130917
3,yolo_s640_seed42,phone,low_light,12,43,12,0.000115,0.000576
4,yolo_s960_seed42,phone,normal,28,123,17,0.075023,0.108072
5,yolo_s960_seed42,phone,low_light,12,43,12,0.000357,0.001180
6,maskrcnn_s320_seed42,phone,normal,28,123,17,0.097545,0.151543
7,maskrcnn_s320_seed42,phone,low_light,12,43,12,0.036860,0.081408
8,maskrcnn_s640_seed42,phone,normal,28,123,17,0.107436,0.175131
9,maskrcnn_s640_seed42,phone,low_light,12,43,12,0.054789,0.116971


## Next step

Send back the number of failed jobs and the result table. The tables and domain-shift analysis are built from `results/eval/` afterwards.